In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, year, to_date, explode, split, regexp_replace, lower, trim, count, desc, row_number, concat
from pyspark.sql.window import Window

import time as tm
from time import time
import sys

In [2]:
# 1. Создаём SparkSession (если ещё не создана)
# spark = SparkSession.builder \
#     .appName("Top10ProgrammingLanguages") \
#     .config("spark.jars.packages", "com.databricks:spark-xml_2.12:0.14.0") \
#     .getOrCreate()

spark = SparkSession.builder \
    .appName("Top10ProgrammingLanguages") \
    .master("local[*]") \
    .config("spark.jars.packages", "com.databricks:spark-xml_2.12:0.14.0") \
    .config("spark.ui.enabled", "true") \
    .config("spark.ui.port", "4040") \
    .config("spark.sql.repl.eagerEval.enabled", "true") \
    .getOrCreate()


spark.sparkContext.setLogLevel("INFO")

# Выводим URL веб-интерфейса в лог (полезно для проверки)
print(f"Spark UI доступен по адресу: {spark.sparkContext.uiWebUrl}")

# print(df_posts.rdd.getNumPartitions())   # количество партиций
# df_counts.explain()                       # план выполнения

Spark UI доступен по адресу: http://52ff20bebd74:4040


In [3]:
# 2. Загружаем XML-файл с постами StackOverflow

# Функция для вывода прогресса с временными метками
def log_progress(message, start_time=None):
    elapsed = f" (за {time()-start_time:.1f} с)" if start_time else ""
    print(f"[{tm.strftime('%H:%M:%S')}] {message}{elapsed}", flush=True)

start_total = time()
log_progress("Начало загрузки XML...")
xml_path = "/home/jovyan/work/DATA/Posts.xml"

spark.sparkContext.setJobDescription("Чтение XML и парсинг")
df_posts = spark.read.format("xml") \
    .option("rowTag", "row") \
    .load(xml_path)

log_progress("XML загружен", start_total)

# Переименовываем столбцы, убирая ведущее подчёркивание
start_rename = time()
log_progress("Переименование столбцов...")
for col_name in df_posts.columns:
    if col_name.startswith("_"):
        df_posts = df_posts.withColumnRenamed(col_name, col_name[1:])
log_progress("Столбцы переименованы", start_rename)

# Выбираем нужные колонки и извлекаем год из CreationDate
start_select = time()
log_progress("Извлечение года из CreationDate...")
df_posts = df_posts.select(
    col("CreationDate"),
    col("Tags")
).withColumn("Year", year(to_date(col("CreationDate"))))
log_progress("Год извлечён", start_select)

# Фильтруем только нужные годы (2010–2020)
start_filter = time()
log_progress("Фильтрация по годам 2010-2020...")
spark.sparkContext.setJobDescription("Фильтрация по годам")
df_posts = df_posts.filter((col("Year") >= 2010) & (col("Year") <= 2020))

# Принудительно вычисляем количество записей, чтобы увидеть прогресс в логах
rows_count = df_posts.count()
log_progress(f"Фильтрация завершена. Оставлено записей: {rows_count:,}", start_filter)

log_progress(f"Общее время выполнения блока: {time()-start_total:.1f} с")

[06:55:36] Начало загрузки XML...
[07:23:53] XML загружен (за 1697.1 с)
[07:23:53] Переименование столбцов...
[07:23:53] Столбцы переименованы (за 0.2 с)
[07:23:53] Извлечение года из CreationDate...
[07:23:53] Год извлечён (за 0.1 с)
[07:23:53] Фильтрация по годам 2010-2020...
[07:39:11] Фильтрация завершена. Оставлено записей: 49,275,038 (за 918.5 с)
[07:39:11] Общее время выполнения блока: 2615.9 с


In [4]:
rows_count

49275038

In [5]:
# 3. Загружаем список языков программирования
langs_path = "/home/jovyan/work/Users/musai/Downloads/programming-languages.csv"
df_langs = spark.read.option("header", True).csv(langs_path)
df_langs = df_langs.withColumn("lang_lower", lower(trim(col("name"))))

In [6]:
# 4. Извлекаем теги из строки Tags
#    Теги хранятся в виде "<tag1><tag2>...". Извлекаем все подстроки между '<' и '>'
#    Используем regexp_extract_all (Spark 3+), в более старых версиях потребуется UDF

# df_tags = df_posts.withColumn(
#     "TagsCleaned", 
#     regexp_replace(col("Tags"), "^<|>$", "")  # убираем первый < и последний >
# ).withColumn(
#     "TagArray", 
#     split(col("TagsCleaned"), "><")
# ).select(
#     col("Year"),
#     explode(col("TagArray")).alias("Tag")
# )

df_tags = df_posts.withColumn(
    "TagArray",
    split(col("Tags"), "\\|")                     # режем по |
).withColumn(
    "Tag", explode(col("TagArray"))
).filter(
    col("Tag") != ""                              # отбрасываем пустые строки (первый и последний элемент)
).select("Year", "Tag")

# Приводим теги к нижнему регистру и удаляем лишние пробелы
df_tags = df_tags.withColumn("Tag", lower(trim(col("Tag"))))

In [7]:
# 5. Соединяем теги со списком языков (inner join) – остаются только теги, являющиеся языками
df_lang_mentions = df_tags.join(df_langs, df_tags.Tag == df_langs.lang_lower, "inner")

In [8]:
# 6. Группируем по году и языку, считаем количество упоминаний (постов)
df_counts = df_lang_mentions.groupBy("Year", "name").agg(count("*").alias("PostCount"))

In [9]:
# 7. Для каждого года выбираем топ-10 языков по количеству упоминаний
window_spec = Window.partitionBy("Year").orderBy(desc("PostCount"))
df_top10 = df_counts.withColumn("rank", row_number().over(window_spec)) \
    .filter(col("rank") <= 10) \
    .orderBy("Year", "rank")

In [10]:
from pyspark.sql.functions import collect_list, concat_ws, lit


# Формируем строку вида "Java (52), PHP (46), ..." для каждого года
df_grouped = df_top10.orderBy("Year", "rank") \
    .withColumn("lang_stat", concat(col("name"), lit(" ("), col("PostCount"), lit(")"))) \
    .groupBy("Year") \
    .agg(concat_ws(", ", collect_list("lang_stat")).alias("Top10_Languages"))

# Показываем результат
df_grouped.orderBy("Year").show(100, truncate=False)

+----+------------------------------------------------------------------------------------------------------------------------------------------------------------+
|Year|Top10_Languages                                                                                                                                             |
+----+------------------------------------------------------------------------------------------------------------------------------------------------------------+
|2010|Java (54220), PHP (51109), JavaScript (43448), Python (26950), Objective-C (17870), C (15231), Ruby (10150), Perl (4999), T-SQL (3893), Delphi (3859)       |
|2011|Java (98291), PHP (94826), JavaScript (89483), Python (41970), Objective-C (38522), C (22532), Ruby (18639), Perl (6724), R (5821), Delphi (5365)           |
|2012|Java (143714), JavaScript (135537), PHP (130044), Python (63920), Objective-C (46407), C (30837), Ruby (24235), R (12157), Bash (8279), Perl (7717)         |
|2013|JavaScript

In [11]:
# Конвертируем в Pandas и показываем
df_pandas = df_top10.select("Year", "rank", "name", "PostCount") \
                              .orderBy("Year", "rank") \
                              .toPandas()

# Отображаем в виде сводной таблицы
pivot_table = df_pandas.pivot(index="Year", columns="rank", values="name")
pivot_table

rank,1,2,3,4,5,6,7,8,9,10
Year,,,,,,,,,,
2010,Java,PHP,JavaScript,Python,Objective-C,C,Ruby,Perl,T-SQL,Delphi
2011,Java,PHP,JavaScript,Python,Objective-C,C,Ruby,Perl,R,Delphi
2012,Java,JavaScript,PHP,Python,Objective-C,C,Ruby,R,Bash,Perl
2013,JavaScript,Java,PHP,Python,Objective-C,C,Ruby,R,Bash,MATLAB
2014,JavaScript,Java,PHP,Python,Objective-C,C,R,Ruby,Bash,MATLAB
2015,JavaScript,Java,PHP,Python,R,C,Objective-C,Ruby,Bash,MATLAB
2016,JavaScript,Java,PHP,Python,R,C,Ruby,Objective-C,Bash,Scala
2017,JavaScript,Python,Java,PHP,R,C,TypeScript,Ruby,Bash,Scala
2018,JavaScript,Python,Java,PHP,R,TypeScript,C,Bash,Scala,Ruby
